# 0. Settings

Import the libraries, packages and files required

In [21]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import json

In [20]:
DATA_DIR = Path("data")
GPKG_PATH = DATA_DIR / "snapshot.gpkg"

# 1. Attribute completeness

### Requirements

1. **Settings:** the required attributes list per class.
2. **Check each rule per feature:** one column per rule where True or False values are stored if the rule applies. For example has_name. Values like surface or religion are checked in the tags column, which holds several feature's tags.
3. **Feature score:** the share of rules it passes, from 0 to 1. A road with a name but no surface gets 0.5.
4. **Class score:** the average of its feature scores. This is the S for attribute completeness in the DQI in equation (1).
5. **Save the result:** a completeness table (feature_id, rule columns, score) and the errors (feature_id plus the rules it failed), so there's easy access or way to filter the failing features in QGIS.

## 1.1 Defining rules and calculate scores per features

### 1.1.1 Roads

#### 1.1.1.1 Defining the attribute list per class and the rules

In [15]:
# MAIN_ROADS = ["motorway", "motorway_link", "trunk", "trunk_link",
#               "primary", "primary_link", "secondary", "secondary_link",
#               "tertiary", "tertiary_link"]

MAIN_ROADS = ["motorway", "motorway_link", "trunk", "trunk_link",
              "primary", "primary_link", "secondary", "secondary_link",
              "tertiary", "tertiary_link", "unclassified", "residential"]

RULES = {
    "roads": [
        {"rule": "has_name",    "tag": "name",    "applies_to": {"highway": MAIN_ROADS}},
        {"rule": "has_surface", "tag": "surface", "applies_to": None},
    ],
    "buildings": [
        {"rule": "has_building_type", "tag": "building", "applies_to": None, "missing_values": ["yes"]},
    ],
    "pois": [
        {"rule": "has_name",     "tag": "name",     "applies_to": None},
        {"rule": "has_religion", "tag": "religion", "applies_to": {"amenity": ["place_of_worship"]}},
    ],
}

#### 1.1.1.2 Defining the functions to check the rules for roads

In [11]:
def check_rule(tags, rule):
    
    conditions = rule["applies_to"] # check if the rule applies to this feature
    if conditions is not None:
        for key, allowed_values in conditions.items():
            feature_value = tags.get(key)
            if feature_value not in allowed_values:
                return None  # rule does not apply (e.g: no name, surface or religion expected)

    
    value = tags.get(rule["tag"]) # check if the tag is present
    if value is None:
        return False  # tag is not present (if it's not, it is because there wasnt a value there, so it penalizes that row)


    if "missing_values" in rule: # is the value one that counts as missing (e.g. building=yes)?
        if value in rule["missing_values"]:
            return False # in this case, it doesnt provide more information about the geometry, it just says it's a building


    return True # If it passess the previuos checks, that means that the tag present with a useful value, so it doesnt penalize the score

#### 1.1.1.3 Checking the roads

In [18]:
roads = gpd.read_file(GPKG_PATH, layer="roads")
tags = roads["tags"].apply(json.loads)

rule_cols = []
for rule in RULES["roads"]:
    roads[rule["rule"]] = pd.array([check_rule(t, rule) for t in tags], dtype="boolean")
    rule_cols.append(rule["rule"])

roads["passed"] = roads[rule_cols].sum(axis=1)
roads["applicable"] = roads[rule_cols].notna().sum(axis=1)
roads["score"] = roads["passed"] / roads["applicable"]
print(roads[rule_cols + ["score"]].describe(include="all"))
print("Class score (roads):", roads["score"].mean())

       has_name has_surface     score
count      5572        6362    6362.0
unique        2           2      <NA>
top       False       False      <NA>
freq       5524        6073      <NA>
mean        NaN         NaN  0.032144
std         NaN         NaN  0.143877
min         NaN         NaN       0.0
25%         NaN         NaN       0.0
50%         NaN         NaN       0.0
75%         NaN         NaN       0.0
max         NaN         NaN       1.0
Class score (roads): 0.03214397988054071


In [19]:
type = roads.groupby("highway")["has_name"].agg(["count", "sum"])
type["share"] = type["sum"] / type["count"]
print(type)

                count  sum     share
highway                             
footway             0    0      <NA>
living_street       0    0      <NA>
primary            58    0       0.0
primary_link       10    0       0.0
residential      3827    4  0.001045
secondary         145   33  0.227586
secondary_link      7    0       0.0
service             0    0      <NA>
tertiary          314   11  0.035032
tertiary_link       1    0       0.0
track               0    0      <NA>
unclassified     1210    0       0.0


#### 1.1.1.4 Calculating the roads score

### 1.1.2 Polygons

### 1.1.3 POIs

## 1.2 Class score

## 1.3 Save the result for attribute completeness